# Designing the Control Logic

## 1. How Serial Works

**Serial** is a simple, one-byte-at-a-time channel between the Pi and Arduino over a USB cable.

```
Raspberry Pi  ──── USB ────  Arduino
  (Python)                  (C++/sketch)
  sends command  ────►  moves servo
  reads status   ◄────  reports done
```

Both sides must agree on three things before a single byte is exchanged:

| Parameter | Our value | What it means |
|-----------|-----------|---------------|
| **Port**  | `/dev/ttyACM0` | The file the OS assigns to the USB connection |
| **Baud rate** | `115200` | Bits per second — must match the Arduino sketch |
| **Timeout** | `0.1 – 1 s` | How long `readline()` waits before giving up |

### Connecting to serial

```python
import serial, time

def setup_serial(port="/dev/ttyACM0", baud=115200, timeout=0.1):
    ser = serial.Serial(port, baud, timeout=timeout)
    time.sleep(3)            # wait for Arduino to reboot after the USB DTR reset
    ser.reset_input_buffer() # discard any startup noise from the Arduino
    return ser
```

Opening the serial port triggers a hardware reset on the Arduino — it reboots and prints its startup messages.  
The `time.sleep(3)` lets that finish so the first command you send lands on a clean, ready Arduino.  
`reset_input_buffer()` then discards any startup noise that accumulated during the wait.

### Sending a command

```python
def send_command(ser, command):
    ser.write(f"{command}\n".encode())
    ser.flush()
```

1. Format the integer as a string and append `\n` (the Arduino uses this as the message terminator).
2. `.encode()` converts the string to raw bytes.
3. `.flush()` pushes the bytes out immediately instead of waiting for a buffer to fill.

### Receiving a reply

```python
def read_serial_line(ser):
    raw = ser.readline()          # blocks until '\n' or timeout
    if not raw:
        return None               # timeout — nothing came back
    return raw.decode("utf-8", errors="replace").strip()
```

`readline()` collects bytes until it sees `\n`, then returns the whole line.

### Our protocol

Below shows the log from `scripts/test_motor_command.py`:

```
connected: /dev/ttyACM0 @ 115200
enter a motor position/command, or q to quit
go to> 0
sent: 0
received: 'Start homing...'
received: 'Limit switch triggered'
received: 'Homing done, pointer = 0'
received: 'Ready'
received: 'Control: 0 / 1 / 2 / 3'
received: 'LCD cmd: 1:text   or   2:text'
received: 'Re-homing...'
received: 'Start homing...'
received: 'Already at limit, backing off...'
received: 'Limit switch triggered'
received: 'Homing done, pointer = 0'
go to> 1
sent: 1
received: 'Move to: 8500'
received: 'Arrived, pointer = 8500'
received: 'Dumping...'
received: 'Servo back to level'
go to> 2
sent: 2
received: 'Move to: 23000'
received: 'Arrived, pointer = 23000'
received: 'Dumping...'
received: 'Servo back to level'
go to> 3
sent: 3
received: 'Move to: 37000'
received: 'Arrived, pointer = 37000'
received: 'Dumping...'
received: 'Servo back to level'
go to> 0
sent: 0
received: 'Re-homing...'
received: 'Start homing...'
```

> From the serial log, what message tells us the machine has completed its current action and is ready for the next command?

## 2. How PiCamera2 Works

**Picamera2** is the Python library that drives the Raspberry Pi camera module.  
Unlike a webcam, it uses a dedicated hardware pipeline — you configure it once, start it, then pull frames on demand.

```
 ┌─────────────┐   configure   ┌──────────────┐   capture_array   ┌─────────────┐
 │  Picamera2  │ ──────────── ▶│   pipeline   │ ─────────────────▶│ NumPy array │
 │  (object)   │               │  640×480 RGB │                    │ shape HxWx3 │
 └─────────────┘               └──────────────┘                    └─────────────┘
```

The frame comes back as a plain NumPy array (height × width × 3 RGB bytes), so it plugs straight into OpenCV and PIL with no conversion overhead.

---

### Helper blocks

Each function below does exactly one thing.  The final control script is assembled by calling them in sequence.

#### Block A — set up the camera

```python
from picamera2 import Picamera2

def setup_camera(width=640, height=480):
    picam2 = Picamera2()
    picam2.configure(
        picam2.create_preview_configuration(
            main={"size": (width, height), "format": "RGB888"}
        )
    )
    picam2.start()
    return picam2
```

Call this once at startup.  `RGB888` gives three bytes per pixel in R-G-B order, which both OpenCV and PIL understand directly.

#### Block B — capture one frame

```python
def capture_frame(picam2):
    return picam2.capture_array()   # returns ndarray, shape (H, W, 3)
```

Calling this in a loop gives you a live video feed one frame at a time.

#### Block C — display a frame (debug only)

```python
import cv2

def show_frame(frame, title="Camera"):
    cv2.imshow(title, frame)
    return cv2.waitKey(1) & 0xFF   # returns the key pressed (if any)
```

Use this while building your logic to see what the camera actually sees.  Press **q** (`ord('q') == 113`) to break out.

#### Block D — convert frame to PIL Image

```python
from PIL import Image

def to_pil(frame):
    return Image.fromarray(frame)
```

The Gemini API (and most vision models) accept a PIL `Image` object rather than a raw NumPy array.  This one-liner bridges the two worlds.

## 3. Challenge — Assemble the Workflow
### All available blocks at a glance

These are every building block available across the serial and camera sections.  The classification block is shown in two flavours — your final script will use one or the other, not both.

| # | Block | Returns | Covered in |
|---|-------|---------|------------|
| 1 | `setup_serial(port, baud)` | `ser` — open Serial object | Section 1 |
| 2 | `send_command(ser, cmd)` | — | Section 1 |
| 3 | `read_serial_line(ser)` | `str \| None` | Section 1 |
| 5 | `setup_camera()` | `picam2` — running Picamera2 object | Section 2 (A) |
| 6 | `capture_frame(picam2)` | `ndarray` — one RGB frame | Section 2 (B) |
| 8 | `to_pil(frame)` | `PIL.Image` | Section 2 (D) |
| 9a | `classify_gemini(client, frame)` | `"cls1" \| "cls2" \| "cls3" \| None` | Gemini notebook |
| 9b | `classify_yolo(model, frame)` | `"cls1" \| "cls2" \| "cls3" \| None` | YOLO notebook |